# Task 1.3 — Implementación de métricas de redes con NumPy

En este notebook se implementan las funciones solicitadas utilizando únicamente **NumPy** y la librería estándar de Python.

No se utiliza NetworkX ni ninguna otra librería de grafos.

Se implementan:

- `grado(A)`
- `clustering(A)`
- `distancia_promedio(A)` usando **BFS** desde cada nodo

Finalmente, las funciones se verifican con la matriz de adyacencia del **Task 1.2**.


In [1]:
import numpy as np


## Matriz de adyacencia del Task 1.2


In [2]:
A = np.array([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0],
    [1, 1, 0, 0, 1],
    [0, 1, 0, 0, 1],
    [0, 0, 1, 1, 0]
], dtype=int)

print(A)


[[0 1 1 0 0]
 [1 0 1 1 0]
 [1 1 0 0 1]
 [0 1 0 0 1]
 [0 0 1 1 0]]


## a. Función `grado(A)`

El grado de cada nodo corresponde a la suma de los valores de su fila en la matriz de adyacencia.


In [3]:
def grado(A):
    """Retorna un arreglo con el grado de cada nodo."""
    A = np.asarray(A)
    return np.sum(A, axis=1)


In [4]:
grados = grado(A)
grado_promedio = np.mean(grados)

print("Grado de cada nodo:", grados)
print("Grado promedio:", grado_promedio)


Grado de cada nodo: [2 3 3 2 2]
Grado promedio: 2.4


## b. Función `clustering(A)`

Para cada nodo \(i\):

1. Se obtienen sus vecinos.
2. Se cuenta cuántas conexiones existen entre esos vecinos.
3. Se divide entre el número máximo posible de conexiones entre ellos:

\[
\binom{k_i}{2} = \frac{k_i(k_i-1)}{2}
\]

Si un nodo tiene grado menor que 2, su coeficiente de clustering se toma como 0.


In [5]:
def clustering(A):
    """Retorna un arreglo con el coeficiente de clustering local de cada nodo."""
    A = np.asarray(A)
    n = A.shape[0]
    coeficientes = np.zeros(n, dtype=float)

    for i in range(n):
        vecinos = np.where(A[i] != 0)[0]
        k = len(vecinos)

        if k < 2:
            coeficientes[i] = 0.0
            continue

        conexiones_vecinos = 0

        for a in range(k):
            for b in range(a + 1, k):
                u = vecinos[a]
                v = vecinos[b]

                if A[u, v] != 0:
                    conexiones_vecinos += 1

        posibles = k * (k - 1) / 2
        coeficientes[i] = conexiones_vecinos / posibles

    return coeficientes


In [6]:
coef_clustering = clustering(A)

print("Coeficiente de clustering por nodo:")
for i, c in enumerate(coef_clustering, start=1):
    print(f"Nodo {i}: {c:.6f}")

print("Clustering promedio:", np.mean(coef_clustering))


Coeficiente de clustering por nodo:
Nodo 1: 1.000000
Nodo 2: 0.333333
Nodo 3: 0.333333
Nodo 4: 0.000000
Nodo 5: 0.000000
Clustering promedio: 0.3333333333333333


## c. BFS y función `distancia_promedio(A)`

Se implementa una búsqueda en anchura (**BFS**) desde cada nodo para encontrar la distancia geodésica mínima hacia los demás nodos.

Si la red no es conexa, únicamente se consideran los pares de nodos para los cuales existe un camino.


In [7]:
def bfs_distancias(A, origen):
    """
    Ejecuta BFS desde el nodo 'origen' y retorna un arreglo
    con las distancias mínimas hacia todos los nodos.

    Los nodos no alcanzables conservan distancia -1.
    """
    A = np.asarray(A)
    n = A.shape[0]

    distancias = np.full(n, -1, dtype=int)
    distancias[origen] = 0

    cola = [origen]
    inicio = 0

    while inicio < len(cola):
        actual = cola[inicio]
        inicio += 1

        vecinos = np.where(A[actual] != 0)[0]

        for vecino in vecinos:
            if distancias[vecino] == -1:
                distancias[vecino] = distancias[actual] + 1
                cola.append(vecino)

    return distancias


In [8]:
def distancia_promedio(A):
    """
    Calcula la distancia promedio utilizando BFS desde cada nodo.

    Solo se consideran pares conectados y cada par se cuenta
    una sola vez, con i < j.
    """
    A = np.asarray(A)
    n = A.shape[0]

    suma_distancias = 0
    pares_conectados = 0

    for i in range(n):
        distancias = bfs_distancias(A, i)

        for j in range(i + 1, n):
            if distancias[j] != -1:
                suma_distancias += distancias[j]
                pares_conectados += 1

    if pares_conectados == 0:
        return 0.0

    return suma_distancias / pares_conectados


In [9]:
d_promedio = distancia_promedio(A)
print("Distancia promedio:", d_promedio)


Distancia promedio: 1.4


## Matriz completa de distancias

La siguiente celda permite visualizar las distancias geodésicas obtenidas mediante BFS desde cada nodo.


In [10]:
matriz_distancias = np.vstack([
    bfs_distancias(A, i) for i in range(A.shape[0])
])

print("Matriz de distancias geodésicas:")
print(matriz_distancias)


Matriz de distancias geodésicas:
[[0 1 1 2 2]
 [1 0 1 1 2]
 [1 1 0 2 1]
 [2 1 2 0 1]
 [2 2 1 1 0]]


## Verificación automática contra los resultados manuales


In [11]:
grados_esperados = np.array([2, 3, 3, 2, 2])
clustering_esperado = np.array([1.0, 1/3, 1/3, 0.0, 0.0])
distancia_esperada = 1.4

assert np.array_equal(grado(A), grados_esperados)
assert np.allclose(clustering(A), clustering_esperado)
assert np.isclose(distancia_promedio(A), distancia_esperada)

print("Todas las verificaciones fueron exitosas.")
print("Los resultados coinciden con los cálculos manuales del Task 1.2.")


Todas las verificaciones fueron exitosas.
Los resultados coinciden con los cálculos manuales del Task 1.2.


## Resumen de resultados

Para la matriz del Task 1.2 se obtiene:

- Grados: \([2,3,3,2,2]\)
- Grado promedio: \(2.4\)
- Clustering local: \([1,\frac{1}{3},\frac{1}{3},0,0]\)
- Clustering promedio: \(\frac{1}{3}\approx 0.3333\)
- Distancia promedio: \(1.4\)

Estos valores coinciden con los resultados obtenidos manualmente en el Task 1.2.
